In [0]:
import os
import mlflow

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"

model = mlflow.spark.load_model("models:/teams.sensorx.mlp_fault_prediction/1")

In [0]:
from pyspark.ml.feature import StandardScaler
from pyspark.ml.functions import vector_to_array
import pyspark.sql.functions as F

# Read test data and normalize (model was trained on normalized features)
train = spark.read.table("teams.sensorx.train_features_delta")
test = spark.read.table("teams.sensorx.test_features_delta")

scaler = StandardScaler(inputCol="features", outputCol="features_scaled", withMean=True, withStd=True)
scaler_model = scaler.fit(train)
test_norm = scaler_model.transform(test).drop("features").withColumnRenamed("features_scaled", "features")

# Filter to latest 3 weeks
max_date = test_norm.agg(F.max("timeStamp")).collect()[0][0]
test_recent = test_norm.filter(F.col("timeStamp") >= F.date_sub(F.lit(max_date), 21))

# Run predictions
predictions = model.transform(test_recent)

# Join with original table to get actual fault state and deviceId
original = spark.read.table("teams.sensorx.df_sx_800_with_delta").select("timeStamp", "payload_fault_state", "properties_deviceId")

pred_with_fault = predictions.select(
    "timeStamp", "label", "prediction"
).join(original, on="timeStamp", how="left") \
 .withColumn("date", F.to_date("timeStamp"))

# One prediction per day per device
daily_results = pred_with_fault.groupBy("date", "properties_deviceId").agg(
    F.max("prediction").cast("int").alias("daily_prediction"),
    F.max(F.col("payload_fault_state").cast("int")).alias("actual_fault_state"),
    F.max("label").alias("horizon_failure"),
    F.count("*").alias("readings_count")
).orderBy("date", "properties_deviceId")

display(daily_results)

In [0]:
from pyspark.sql import Row

data = [
    Row(Trial=1,  AUC=0.6525), #Logistic Regression 1, 5.mars
    Row(Trial=2,  AUC=0.6251), #Random Forest 1, 5.mars
    Row(Trial=3,  AUC=0.5383), #Gradient Boosted Trees 1, 5.mars
    Row(Trial=4,  AUC=0.9115), #Logistic Regression 2, 8.mars
    Row(Trial=5,  AUC=0.9139), #Logistic Regression 3, 8.mars
    Row(Trial=6,  AUC=0.9147), #Gradient Boosted Trees 2, 8.mars
    Row(Trial=7,  AUC=0.9114), #Logistic Regression 4, 12.mars
    Row(Trial=8,  AUC=0.9005), #Logistic Regression 5, 12.mars
    Row(Trial=9,  AUC=0.9237), #Logistic Regression 6, 12.mars
    Row(Trial=10, AUC=0.9350), #Logistic Regression 7, 12.mars
    Row(Trial=11, AUC=0.9411), #Random Forest 2, 12.mars
    Row(Trial=12, AUC=0.9156), #Gradient Boosted Trees 3, 12.mars
    Row(Trial=13, AUC=0.9240), #Random Forest 3, 15.mars
    Row(Trial=14, AUC=0.9360), #Gradient Boosted Trees 5, 17.mars
    Row(Trial=15, AUC=0.9379), #Multilayer Perceptron 1, 23.mars
    Row(Trial=16, AUC=0.9238), #Multilayer Perceptron 2, 29.mars
    Row(Trial=17, AUC=0.9136), #Multilayer Perceptron 3, 29.mars
    Row(Trial=18, AUC=0.9137), #Multilayer Perceptron 4, 29.mars
    Row(Trial=19, AUC=0.9034), #Multilayer Perceptron 5, 29.mars
    Row(Trial=20, AUC=0.8837), #Multilayer Perceptron 6, 29.mars
    Row(Trial=21, AUC=0.8847), #Multilayer Perceptron 7, 29.mars
    Row(Trial=22, AUC=0.9314), #Multilayer Perceptron 8, 29.mars
    Row(Trial=23, AUC=0.9452), #Multilayer Perceptron 9, 2.apríl
]

df_auc = spark.createDataFrame(data)
display(df_auc)

Databricks visualization. Run in Databricks to view.

In [0]:
import plotly.graph_objects as go

rows = df_auc.orderBy("Trial").collect()
trials = [r.Trial for r in rows]
aucs = [r.AUC for r in rows]

fig = go.Figure()
fig.add_trace(go.Scatter(x=trials, y=aucs, mode="lines+markers", name="AUC", line=dict(color="#919191")))

# Lines with information about where LR and MLP stop/start
fig.add_vline(x=10, line_dash="dash", line_color="#BF7080",
              annotation_text="LR testing stops", annotation_position="top left")
fig.add_vline(x=16, line_dash="dash", line_color="#99DDB4",
              annotation_text="MLP testing starts", annotation_position="top left")

fig.update_layout(
    title="AUC vs Trials",
    xaxis_title="Trial",
    yaxis_title="AUC",
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="lightgray"),
    yaxis=dict(showgrid=True, gridcolor="lightgray"),
)
fig.show()


## MLP Fault Prediction Model — Results & Evaluation
This section presents the evaluation of the **Multilayer Perceptron (MLP)** classifier trained to predict X-ray controller faults using sensor telemetry data. The model uses a **failure horizon** label (fault within the next N hours) rather than the instantaneous fault state, enabling **early warning** capability.

In [0]:
from pyspark.sql import functions as F, Window
import pandas
from datetime import datetime, timedelta
from pyspark.sql.window import Window
import os
import mlflow
from marel.services import ServiceProvider
from marel.databricks import DatabricksProvider
provider = DatabricksProvider()

In [0]:
df_csv = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("multiLine", True) \
    .option("escape", "\"") \
    .option("quote", "\"") \
    .option("delimiter", ",") \
    .load("/Volumes/teams/sensorx/data-dump/deviceID_Serial_GenSerial_machineCode.csv")

# Device IDs that have at least one 800W serial number
devices_800w = (
    df_csv.filter(F.col("gentype") == "800W")
    .select("properties_deviceId")
    .distinct()
)

devices_500w = (
    df_csv.filter(F.col("gentype") == "500W")
    .select("properties_deviceId")
    .distinct()
)

devices_800w.write.format("delta").mode("overwrite").saveAsTable("teams.sensorx.devices_800w")

devices_500w.write.format("delta").mode("overwrite").saveAsTable("teams.sensorx.devices_500w")

In [0]:
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder

def process_data(months):
    df_sx_telemetry = spark.table('marel_digital_machines.sensorx.marel_sensorx_telemetry')
    df_sx_telemetry = df_sx_telemetry.filter(F.col('timeStamp') > F.add_months(F.current_date(),-months)).select(
        "timeStamp",
        "properties_deviceId",
        "payload_xrayController_filamentCurrent",
        "payload_xrayController_temperature",
        "payload_xrayController_tubeCurrent",
        "payload_xrayController_tubeVoltage",
        "payload_xrayController_onTime"
    )

    devices_800w= spark.table("teams.sensorx.devices_800w")

    # Filter telemetry to only 800W devices (broadcast — devices_800w is small)
    df_sx_telemetry = df_sx_telemetry.join(
        F.broadcast(devices_800w), "properties_deviceId", "inner"
    )

    df_sx_xraycontroller_fault = (
        spark.table('marel_digital_machines.sensorx.marel_sensorx_xraycontroller_property_fault')
        .filter(
            (F.col('timeStamp') > F.add_months(F.current_date(), -months)) &
            (F.col('payload_fault_faultName') == "faultRegulation")
        )
        .select("timeStamp", "properties_deviceId", "payload_fault_faultName", "payload_fault_state")
    )

    # Filter faults to 800W devices only
    df_sx_xraycontroller_fault = df_sx_xraycontroller_fault.join(
        F.broadcast(devices_800w), "properties_deviceId", "inner"
    )

    # --- Union + forward-fill ---
    telem_only = [c for c in df_sx_telemetry.columns
                if c not in ("timeStamp", "properties_deviceId")]

    telem_part = df_sx_telemetry.select(
        "timeStamp", "properties_deviceId",
        *telem_only,
        F.lit(None).cast("string").alias("payload_fault_faultName"),
        F.lit(None).cast("boolean").alias("payload_fault_state"),
        F.lit(True).alias("_is_telem"),
    )

    fault_part = df_sx_xraycontroller_fault.select(
        "timeStamp", "properties_deviceId",
        *[F.lit(None).cast(dict(df_sx_telemetry.dtypes)[c]).alias(c)
        for c in telem_only],
        "payload_fault_faultName",
        "payload_fault_state",
        F.lit(False).alias("_is_telem"),
    )

    w = Window.partitionBy("properties_deviceId").orderBy("timeStamp")

    df = (
        telem_part.unionByName(fault_part)
        .withColumn("payload_fault_faultName",
                    F.last("payload_fault_faultName", ignorenulls=True).over(w))
        .withColumn("payload_fault_state",
                    F.last("payload_fault_state", ignorenulls=True).over(w))
        .filter(F.col("_is_telem"))
        .drop("_is_telem")
    )

    df = df.withColumn(
        "payload_fault_state", F.coalesce(F.col("payload_fault_state"), F.lit(False))
    ).withColumn(
        "payload_fault_faultName", F.coalesce(F.col("payload_fault_faultName"), F.lit("faultRegulation"))
    )

    # Compute delta_seconds
    order_cols = [F.col("timeStamp").asc()]
    if "serialNumber" in df.columns:
        order_cols.append(F.col("serialNumber").asc())
    w = Window.partitionBy("properties_deviceId").orderBy(*order_cols)

    df = (
        df
        .withColumn("prev_ts", F.lag("timeStamp").over(w))
        .withColumn(
            "delta_seconds",
            F.when(F.col("prev_ts").isNull(), F.lit(None).cast("double"))
            .otherwise(
                F.when(
                    (F.col("timeStamp").cast("long") - F.col("prev_ts").cast("long")) < 0,
                    None
                ).otherwise(F.col("timeStamp").cast("long") - F.col("prev_ts").cast("long"))
            )
        )
        .drop("prev_ts")
    )

    df = df.drop("serialNumber", "payload_fault_faultName", "GeneratorType")

    # ---- Feature engineering (matching (Clone) model_testing) ----

    # Sensor columns = everything except identifiers and target
    sensor_cols = [c for c in df.columns
                   if c not in {"properties_deviceId", "timeStamp", "payload_fault_state"}]

    # Drop rows with null sensor values
    df = df.na.drop(subset=sensor_cols)

    # One-Hot Encode properties_deviceId
    indexer = StringIndexer(inputCol="properties_deviceId", outputCol="deviceId_index")
    df_indexed = indexer.fit(df).transform(df)

    encoder = OneHotEncoder(inputCol="deviceId_index", outputCol="deviceId_ohe")
    df_encoded = encoder.fit(df_indexed).transform(df_indexed)

    # Lag features (n_lags = 20)
    n_lags = 20
    w_lag = Window.partitionBy("properties_deviceId").orderBy("timeStamp")

    for L in range(1, n_lags + 1):
        for col_name in sensor_cols:
            df_encoded = df_encoded.withColumn(f"{col_name}{L}", F.lag(col_name, L).over(w_lag))

    lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]

    # Drop rows with null lag values
    df_clean = df_encoded.na.drop(subset=lag_cols)

    # Assemble features: sensor + lag + OHE device ID
    feature_input_cols = sensor_cols + lag_cols + ["deviceId_ohe"]
    assembler = VectorAssembler(inputCols=feature_input_cols, outputCol="features")
    df_features = assembler.transform(df_clean)

    df_features = df_features.select("timeStamp", "properties_deviceId", "features")

    return df_features

In [0]:
months = 3

df = process_data(months)
display(df.limit(10))

In [0]:
from pyspark.ml.feature import StandardScaler
from pyspark.ml.functions import vector_to_array

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"

# Read existing test/train data
train = spark.read.table("teams.sensorx.train_features_delta")
test = spark.read.table("teams.sensorx.test_features_delta")

# Normalize features (fit scaler on training data to avoid data leakage)
scaler = StandardScaler(inputCol="features", outputCol="features_scaled", withMean=True, withStd=True)
scaler_model = scaler.fit(train)
test_norm = scaler_model.transform(test).drop("features").withColumnRenamed("features_scaled", "features")

# Load model and run predictions
model = mlflow.spark.load_model("models:/teams.sensorx.mlp_fault_prediction/3")
predictions = model.transform(test_norm)

# Simplify output: extract fault probability and drop vector columns
results = predictions.select(
    "timeStamp",
    F.col("label").alias("actual_label"),
    F.col("prediction").cast("int").alias("predicted_label"),
    F.round(vector_to_array("probability")[1], 4).alias("fault_probability"),
)

display(results)